In [0]:
display(
    dbutils.fs.ls(
        "abfss://bronze@adfassignment7.dfs.core.windows.net/sales_view/"
    )
)

In [0]:
customer_path = "abfss://bronze@adfassignment7.dfs.core.windows.net/sales_view/customer/"

customer_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(customer_path)
)

display(customer_df)

Convert column names to lowercase snake_case

In [0]:
import re

def to_snake_case(column_name):
    column_name = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', column_name)
    column_name = re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', column_name)
    column_name = re.sub(r'[^a-zA-Z0-9]+', '_', column_name)
    return column_name.strip('_').lower()

customer_df = customer_df.toDF(
    *[to_snake_case(c) for c in customer_df.columns]
)

display(customer_df)

In [0]:
# Step 2: Split name into first_name and last_name

from pyspark.sql.functions import col, split, trim

customer_df = (
    customer_df
    .withColumn("first_name", trim(split(col("name"), " ").getItem(0)))
    .withColumn("last_name", trim(split(col("name"), " ").getItem(1)))
)

display(customer_df)

In [0]:
# Step 3: Extract domain from email_id

from pyspark.sql.functions import col, regexp_extract

customer_df = customer_df.withColumn(
    "domain",
    regexp_extract(col("email_id"), r"@([^.]+)", 1)
)

display(customer_df)

In [0]:
# Step 4: Convert gender values to M/F

from pyspark.sql.functions import col, when, lower

customer_df = customer_df.withColumn(
    "gender",
    when(lower(col("gender")) == "male", "M")
    .when(lower(col("gender")) == "female", "F")
    .otherwise(None)
)

display(customer_df)

In [0]:
# Step 5: Split joining_date into date and time

from pyspark.sql.functions import col, split, trim

customer_df = (
    customer_df
    .withColumn("date", trim(split(col("joining_date"), " ").getItem(0)))
    .withColumn("time", trim(split(col("joining_date"), " ").getItem(1)))
)

display(customer_df)

In [0]:
# Check joining_date values

display(
    customer_df
    .select("joining_date")
    .distinct()
    .orderBy("joining_date")
)

In [0]:
# Step 6: Convert date to yyyy-MM-dd

from pyspark.sql.functions import col, split, trim, expr

customer_df = (
    customer_df
    .drop("date", "time")
    .withColumn(
        "date",
        expr("""
            try_to_date(
                trim(split(joining_date, ' ')[0]),
                'dd-MM-yyyy'
            )
        """)
    )
    .withColumn(
        "time",
        trim(split(col("joining_date"), " ").getItem(1))
    )
)

display(customer_df)

In [0]:
# Step 7: Create expenditure_status

from pyspark.sql.functions import col, when

customer_df = customer_df.withColumn(
    "expenditure_status",
    when(col("spent") < 200, "MINIMUM")
    .otherwise("MAXIMUM")
)

display(customer_df)

In [0]:
# Step 8: Write Customer data to Silver

silver_path = "abfss://silver@adfassignment7.dfs.core.windows.net/sales_view/customer/"

customer_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

print("Customer data written successfully to Silver")

In [0]:
#bronze to silver